In [1]:
import os
os.chdir('../')
os.getcwd()

'C:\\Users\\Siddharth\\Desktop\\local\\flipkart\\Round2'

In [2]:
import os
import random
import shutil
from pathlib import Path

import yaml
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from ultralytics import YOLO

In [29]:
# =====================================
# CONFIG
# =====================================

DATASET_ROOT = Path("dataset/licence_plate_dataset/train")

IMAGES_DIR = DATASET_ROOT / "images"
LABELS_DIR = DATASET_ROOT / "labels"

OUTPUT_DATASET = Path("dataset/license_plate_yolo")

TRAIN_SIZE = 5000
VAL_RATIO = 0.10

RANDOM_STATE = 42

MODEL_NAME = "yolov8n.pt"

FINAL_MODEL_PATH = Path("models/license_weights.pt")
FINAL_MODEL_PATH.parent.mkdir(exist_ok=True)

In [4]:
image_files = sorted(list(IMAGES_DIR.glob("*.jpg")))

print(f"Total Images Found: {len(image_files)}")

missing_labels = []

for img_path in image_files:
    label_path = LABELS_DIR / f"{img_path.stem}.txt"

    if not label_path.exists():
        missing_labels.append(img_path.name)

print(f"Missing Labels: {len(missing_labels)}")

if len(missing_labels) > 0:
    print("\nFirst 10 Missing Labels:")
    print(missing_labels[:10])

assert len(missing_labels) == 0, "Dataset contains missing labels!"

Total Images Found: 11805
Missing Labels: 0


In [5]:
# =====================================
# Convert Segmentation Labels To BBoxes
# =====================================

from pathlib import Path

converted = 0
already_detection = 0

for txt_file in LABELS_DIR.glob("*.txt"):

    new_lines = []

    with open(txt_file, "r") as f:

        lines = f.readlines()

    for line in lines:

        values = line.strip().split()

        if len(values) == 0:
            continue

        # Detection label
        if len(values) == 5:

            new_lines.append(line.strip())
            already_detection += 1

        # Segmentation label
        elif len(values) > 5:

            cls_id = values[0]

            coords = list(map(float, values[1:]))

            xs = coords[0::2]
            ys = coords[1::2]

            xmin = min(xs)
            xmax = max(xs)

            ymin = min(ys)
            ymax = max(ys)

            x_center = (xmin + xmax) / 2
            y_center = (ymin + ymax) / 2

            width = xmax - xmin
            height = ymax - ymin

            bbox_line = (
                f"{cls_id} "
                f"{x_center:.6f} "
                f"{y_center:.6f} "
                f"{width:.6f} "
                f"{height:.6f}"
            )

            new_lines.append(bbox_line)

            converted += 1

        else:

            raise ValueError(
                f"Invalid label format in {txt_file}"
            )

    with open(txt_file, "w") as f:

        for l in new_lines:
            f.write(l + "\n")

print(f"Detection Labels Retained : {already_detection}")
print(f"Segmentation Labels Converted : {converted}")

Detection Labels Retained : 13419
Segmentation Labels Converted : 3411


In [6]:
if OUTPUT_DATASET.exists():
    shutil.rmtree(OUTPUT_DATASET)

for split in ["train", "val"]:

    (OUTPUT_DATASET / split / "images").mkdir(parents=True)
    (OUTPUT_DATASET / split / "labels").mkdir(parents=True)

print("YOLO dataset folders created.")

YOLO dataset folders created.


In [7]:
all_images = sorted(list(IMAGES_DIR.glob("*.jpg")))

selected_images = random.sample(
    all_images,
    min(TRAIN_SIZE, len(all_images))
)

train_imgs, val_imgs = train_test_split(
    selected_images,
    test_size=VAL_RATIO,
    random_state=RANDOM_STATE
)

print("Train Images:", len(train_imgs))
print("Val Images:", len(val_imgs))

Train Images: 4500
Val Images: 500


In [8]:
for img_path in train_imgs:

    label_path = LABELS_DIR / f"{img_path.stem}.txt"

    shutil.copy(
        img_path,
        OUTPUT_DATASET / "train" / "images" / img_path.name
    )

    shutil.copy(
        label_path,
        OUTPUT_DATASET / "train" / "labels" / label_path.name
    )

print("Train set copied.")

Train set copied.


In [9]:
for img_path in val_imgs:

    label_path = LABELS_DIR / f"{img_path.stem}.txt"

    shutil.copy(
        img_path,
        OUTPUT_DATASET / "val" / "images" / img_path.name
    )

    shutil.copy(
        label_path,
        OUTPUT_DATASET / "val" / "labels" / label_path.name
    )

print("Validation set copied.")

Validation set copied.


In [10]:
yaml_data = {
    "path": str(OUTPUT_DATASET.resolve()),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": ["licenseplate"]
}

yaml_path = OUTPUT_DATASET / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print("Created:", yaml_path)

Created: dataset\license_plate_yolo\data.yaml


In [11]:
with open(yaml_path, "r") as f:
    print(f.read())

path: C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\license_plate_yolo
train: train/images
val: val/images
nc: 1
names:
- licenseplate



In [12]:
model = YOLO(MODEL_NAME)

print("YOLOv8n loaded.")

YOLOv8n loaded.


In [16]:
results = model.train(
    data=str(yaml_path),

    epochs=20,
    imgsz=512,

    batch=8,

    device="cpu",

    workers=4,

    pretrained=True,

    project=r"C:\Users\Siddharth\Desktop\local\flipkart\Round2\models",
    name="license_plate_detector",

    patience=10,

    optimizer="auto",

    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    fliplr=0.0,
    flipud=0.0,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    verbose=True
)

New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.70  Python-3.10.20 torch-2.12.1+cpu CPU (AMD Ryzen 5 7520U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\license_plate_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_s

In [30]:
trained_model = YOLO(str(FINAL_MODEL_PATH))

metrics = trained_model.val(
    data=str(yaml_path),
    split="val"
)

print(metrics)

Ultralytics 8.4.70  Python-3.10.20 torch-2.12.1+cpu CPU (AMD Ryzen 5 7520U with Radeon Graphics)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.70.9 ms, read: 156.722.8 MB/s, size: 153.7 KB)
val: Scanning C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\license_plate_yolo\val\labels.cache... 500 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.1s/it 36.3s1.1ss
                   all        500        726      0.957      0.839      0.904      0.645
Speed: 1.0ms preprocess, 61.8ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to C:\Users\Siddharth\runs\detect\val-11
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object a

In [31]:
print("\nPrecision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)


Precision : 0.9568320482997384
Recall    : 0.8388429752066116
mAP50     : 0.9036701315283308
mAP50-95  : 0.6446434845073925


In [33]:
sample_image = random.choice(
    list((OUTPUT_DATASET / "val" / "images").glob("*.jpg"))
)

print(sample_image)

results = trained_model.predict(
    source=str(sample_image),
    conf=0.25,
    save=False
)

dataset\license_plate_yolo\val\images\c94k9fqfdhla1_jpg.rf.cc7ac58705f1d43ddf8dd68a4c4fca26.jpg

image 1/1 C:\Users\Siddharth\Desktop\local\flipkart\Round2\dataset\license_plate_yolo\val\images\c94k9fqfdhla1_jpg.rf.cc7ac58705f1d43ddf8dd68a4c4fca26.jpg: 512x512 4 licenseplates, 159.1ms
Speed: 2.9ms preprocess, 159.1ms inference, 3.6ms postprocess per image at shape (1, 3, 512, 512)


In [34]:
print(results)

[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'licenseplate'}
obb: None
orig_img: array([[[186, 171, 228],
        [112,  99, 151],
        [ 31,  19,  67],
        ...,
        [203, 191, 173],
        [174, 162, 144],
        [204, 192, 174]],

       [[103,  88, 145],
        [ 78,  65, 117],
        [ 40,  29,  75],
        ...,
        [213, 201, 183],
        [200, 188, 170],
        [170, 158, 140]],

       [[ 32,  18,  72],
        [ 30,  18,  68],
        [ 50,  39,  85],
        ...,
        [211, 198, 182],
        [193, 181, 163],
        [207, 195, 177]],

       ...,

       [[111, 157, 175],
        [ 91, 137, 155],
        [212, 254, 255],
        ...,
        [133, 141, 131],
        [182, 190, 180],
        [154, 162, 152]],

       [[112, 157, 178],
        [118, 164, 182],
        [105, 147, 166],
        ...,
        [170, 178, 168],
        [186, 194, 184],
      